# CENG 467 Full-Scale Training on Google Colab

Bu notebook tam ölçekli eğitim (Q1-Q5) GPU'da çalıştırır ve **Google Drive**'a kaydeder.

**Kurulum**: 
1. Colab üstte **"GPU"** seç → Runtime → Change runtime type → T4 GPU
2. Google Drive mount edecek (kimlik doğrulama gerekebilir)
3. Outputs otomatik `My Drive/zubeyr-nlp/` klasörüne kaydedilecek

## 1. Setup: Clone Repository & Dependencies

In [ ]:
import os
import subprocess

os.chdir('/content')

if not os.path.exists('467-takehome'):
    print("📦 Cloning repository...")
    subprocess.run(['git', 'clone', 'https://github.com/zubeyralmaho/467-takehome.git'], check=True)
else:
    print("✅ Repository already exists")
    os.chdir('467-takehome')
    subprocess.run(['git', 'pull'], check=False)
    os.chdir('/content')

os.chdir('/content/467-takehome')
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q -r requirements.txt
print("✅ Dependencies installed")

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

drive_dir = '/content/drive/MyDrive/zubeyr-nlp'
os.makedirs(drive_dir, exist_ok=True)
print(f"✅ Google Drive mounted")
print(f"📁 Outputs will be saved to: My Drive/zubeyr-nlp/")

In [ ]:
!nvidia-smi
import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## 2-6. Run Full-Scale Training (Q1-Q5)

In [ ]:
print("\n" + "="*70)
print("Q1: TEXT CLASSIFICATION (IMDb - FULL DATASET)")
print("="*70)
!python -m src.q1_classification.main --config configs/q1_full.yaml --final-eval
print("✅ Q1 complete\n")

In [ ]:
print("\n" + "="*70)
print("Q2: NAMED ENTITY RECOGNITION (CoNLL-2003 - FULL DATASET)")
print("="*70)
!python -m src.q2_ner.main --config configs/q2_full.yaml --final-eval
print("✅ Q2 complete\n")

In [ ]:
print("\n" + "="*70)
print("Q3: TEXT SUMMARIZATION (CNN/DailyMail - FULL DATASET)")
print("="*70)
!python -m src.q3_summarization.main --config configs/q3_full.yaml --final-eval
print("✅ Q3 complete\n")

In [ ]:
print("\n" + "="*70)
print("Q4: MACHINE TRANSLATION (Multi30k EN→DE - FULL DATASET)")
print("="*70)
!python -m src.q4_machine_translation.main --config configs/q4_full.yaml --final-eval
print("✅ Q4 complete\n")

In [ ]:
print("\n" + "="*70)
print("Q5: LANGUAGE MODELING (WikiText-2 - FULL DATASET)")
print("="*70)
!python -m src.q5_language_modeling.main --config configs/q5_full.yaml --final-eval
print("✅ Q5 complete\n")

## 7. Save All Outputs to Google Drive

In [ ]:
import os
import glob
import shutil
import time
from pathlib import Path

print("\n" + "="*70)
print("SAVING ALL OUTPUTS TO GOOGLE DRIVE")
print("="*70)

drive_output_dir = '/content/drive/MyDrive/zubeyr-nlp'

if os.path.exists(drive_output_dir):
    # Create ZIP archive
    print("\n1️⃣  Creating ZIP archive...")
    archive_path = shutil.make_archive('outputs', 'zip', 'outputs')
    archive_size = os.path.getsize(archive_path) / (1024**2)
    print(f"   ✅ {archive_size:.1f} MB")
    
    # Copy to Drive
    timestamp = int(time.time())
    drive_archive = os.path.join(drive_output_dir, f'outputs_{timestamp}.zip')
    shutil.copy(archive_path, drive_archive)
    print(f"   ✅ Saved to Drive")
    
    # Copy individual run directories
    print("\n2️⃣  Copying run directories...")
    for q in range(1, 6):
        runs_dir = f'outputs/q{q}'
        if os.path.exists(runs_dir):
            runs = sorted(glob.glob(os.path.join(runs_dir, 'run_*')))
            if runs:
                latest_run = runs[-1]
                run_name = os.path.basename(latest_run)
                dest_dir = os.path.join(drive_output_dir, f'q{q}_{run_name}')
                
                if os.path.exists(dest_dir):
                    shutil.rmtree(dest_dir)
                
                shutil.copytree(latest_run, dest_dir)
                print(f"   ✅ Q{q}: {run_name}")
    
    print(f"\n" + "="*70)
    print("✅ ALL OUTPUTS SAVED TO: My Drive/zubeyr-nlp/")
    print("="*70)
    
    print(f"\n📂 Available in your Drive:")
    files = sorted(os.listdir(drive_output_dir))
    for f in files:
        path = os.path.join(drive_output_dir, f)
        if os.path.isfile(path):
            size = os.path.getsize(path) / (1024**2)
            print(f"   📦 {f} ({size:.1f} MB)")
        else:
            print(f"   📁 {f}/")
else:
    print("⚠️ ERROR: Drive not mounted!")

## 8. Done! 🎉

In [ ]:
print("\n" + "="*70)
print("🎉 FULL-SCALE TRAINING COMPLETE!")
print("="*70)
print("\n✅ All outputs saved to Google Drive: My Drive/zubeyr-nlp/")
print("\n📊 Generated artifacts:")
print("   • Training curves (PNG) for convergence analysis")
print("   • Metrics JSON for all models")
print("   • Confusion matrices and predictions")
print("   • Error distributions (Q2)")
print("   • Attention heatmaps (Q4)")
print("\n🚀 Next steps:")
print("   1. Download artifacts from Google Drive")
print("   2. Copy figures to report/figures/")
print("   3. Update LaTeX references")
print("   4. Build final report")
print("\n" + "="*70)